# Step 3 - EDA 1D

Looking at each column individually. Goal is to understand the shape of the data,
not to tell a story yet. Some of these will be boring, that's fine.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('../data/cleaned.parquet')
print(f'{len(df):,} rows, {df.shape[1]} cols')
df.dtypes

## Valeur fonciere (main column)

In [ ]:
# only looking at regular sales, no outliers
ventes = df[(df['nature_mutation'] == 'Vente') & (~df['is_outlier_valeur'])]
print(f'Ventes non-outlier: {len(ventes):,}')
ventes['valeur_fonciere'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sample = ventes['valeur_fonciere'].sample(80000, random_state=1)

axes[0].hist(sample, bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('Valeur foncière - Ventes')
axes[0].set_xlabel('€')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# log scale makes it more readable
axes[1].hist(np.log10(sample + 1), bins=80, color='steelblue', edgecolor='none')
axes[1].set_title('Log10(Valeur foncière)')
axes[1].set_xlabel('log10(€)')

plt.tight_layout()
plt.savefig('../outputs/plots/01_valeur_dist.png', dpi=100)
plt.show()

# note: right skewed as expected, log scale shows something closer to normal
# most sales cluster between 50k and 500k

## Type local

In [ ]:
counts = df['type_local'].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Transactions par type de bien')
ax.set_xlabel('')
ax.set_ylabel('Nombre de transactions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/plots/02_type_local.png', dpi=100)
plt.show()

# Terrain dominates - makes sense, a lot of land gets sold separately from buildings
# Dépendance (garages, annexes) is surprisingly high

## Nature mutation

In [ ]:
nm = df['nature_mutation'].value_counts()
print(nm)

# not going to plot this - it's basically all Vente, the chart would be pointless
# 93% are Vente, the rest are VEFA, échange, adjudication

## Transactions per year

In [ ]:
by_year = df.groupby('annee').size()
print(by_year)

fig, ax = plt.subplots(figsize=(7, 4))
by_year.plot(kind='bar', ax=ax, color='coral', edgecolor='none')
ax.set_title('Nombre de transactions par année')
ax.set_xlabel('Année')
ax.set_ylabel('Transactions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/plots/03_transactions_par_an.png', dpi=100)
plt.show()

# 2021 and 2022 were peak years (~4.67M each)
# drop from 2023 onward - matches known market slowdown in France (rate hikes)

## Surface bati

In [ ]:
# only rows with actual buildings
with_bati = df[df['surface_bati'] > 0]['surface_bati']
print(with_bati.describe())
print(f'\n> 500m²: {(with_bati > 500).sum():,}')

fig, ax = plt.subplots(figsize=(8, 4))
with_bati[with_bati <= 500].hist(bins=80, ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Surface bâtie (≤500m²)')
ax.set_xlabel('m²')
plt.tight_layout()
plt.savefig('../outputs/plots/04_surface_bati.png', dpi=100)
plt.show()

# expected more of a peak around 80-100m² for apartments
# the distribution is actually fairly flat up to ~120m²
# makes sense since it mixes houses and apartments

In [ ]:
# the overall surface distribution looked flat - probably because it mixes house and apartment sizes
# let's split by type_local to see if that explains it

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, t, color in zip(axes, ['Maison', 'Appartement'], ['steelblue', 'coral']):
    subset = df[(df['type_local'] == t) & (df['surface_bati'] > 0) & (df['surface_bati'] <= 300)]['surface_bati']
    subset.hist(bins=60, ax=ax, color=color, edgecolor='none')
    ax.set_title(f'Surface bâtie - {t} (n={len(subset):,})')
    ax.set_xlabel('m²')
    med = subset.median()
    ax.axvline(med, color='black', linestyle='--', linewidth=1)
    ax.text(med + 3, ax.get_ylim()[1]*0.9, f'médiane {med:.0f}m²', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/plots/04b_surface_par_type.png', dpi=100)
plt.show()

# this explains the flat distribution: houses peak around 90-100m², apartments around 50-60m²
# mixed together they smooth out the histogram

## Nombre de pièces

In [ ]:
pieces = df[df['nb_pieces'] > 0]['nb_pieces'].value_counts().sort_index()
print(pieces)

fig, ax = plt.subplots(figsize=(8, 4))
pieces[pieces.index <= 15].plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Nombre de pièces principales')
ax.set_xlabel('Pièces')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/plots/05_nb_pieces.png', dpi=100)
plt.show()

# peak at 4 pieces, which makes sense for houses
# 1-2 pieces are studios/small apartments

## Valeur médiane par année

In [ ]:
# median is more robust than mean for price data
ventes_clean = df[(df['nature_mutation'] == 'Vente') & (~df['is_outlier_valeur'])]
med_by_year = ventes_clean.groupby('annee')['valeur_fonciere'].median()
print(med_by_year)

fig, ax = plt.subplots(figsize=(7, 4))
med_by_year.plot(kind='bar', ax=ax, color='coral', edgecolor='none')
ax.set_title('Valeur médiane des ventes par année')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k€'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../outputs/plots/06_valeur_mediane_annee.png', dpi=100)
plt.show()

# interesting to see if prices went up/down over the 5 years

## Top departements par volume

In [ ]:
top_dept = df['code_departement'].value_counts().head(15)
print(top_dept)

fig, ax = plt.subplots(figsize=(10, 4))
top_dept.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Top 15 départements par nombre de transactions')
ax.set_xlabel('Code département')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/plots/07_top_departements.png', dpi=100)
plt.show()

# 75 (Paris), 13 (Bouches-du-Rhône), 69 (Rhône) - makes sense, biggest cities

## Nature culture (land type)

In [ ]:
# nature_culture: counts by category
nc = df['nature_culture'].value_counts()
print(nc.head(10))
print(f'\nNull (no land culture = building rows): {df["nature_culture"].isnull().sum():,}')

# before dismissing this, let's check if land culture type affects price
# agricultural land (L, P, V) vs constructible land (S=sol/land) might differ a lot
nc_price = (
    df[df['nature_culture'].notna() & ~df['is_outlier_valeur']]
    .groupby('nature_culture')['valeur_fonciere']
    .agg(['median', 'count'])
    .query('count > 1000')
    .sort_values('median', ascending=False)
)
print('\nMedian valeur by nature_culture (min 1000 rows):')
print(nc_price)

# S (sol - building land) is the most valuable, agricultural categories much cheaper
# this actually could be useful as a feature for terrain rows